# Stage 2: Adaptive Observation Scheduling Engine

**Research Question:** Can an AI-driven scheduler optimize exoplanet observation allocation more effectively than static prioritization approaches?

---

## Architecture

```
Stage 1 Priority Scores + Uncertainty
        |
        v
[A] Observation Constraint Engine   <- AR(1) weather, time budget, visibility
        |
        v
[B] Scientific Gain Calculator      <- Gain = a*U + b*P + g*D  (b decays over time)
        |
        v
[C] 5 Scheduler Algorithms
    1. Static Priority              (Baseline 1)
    2. Detectability Greedy         (Baseline 2)
    3. Uncertainty Greedy           (Baseline 3)
    4. Adaptive Scheduler           (Our method)
    5. Oracle Scheduler             (Upper bound)
        |
        v
[D] Observation Simulator           <- SNR/weather/detectability-dependent noise
        |
        v
[E] Adaptive Reprioritization Loop  <- 30 rounds x 10 planets
        |
        v
[F] Evaluation & Metrics            <- 7 metrics including Campaign Diversity
        |
        v
[G] Streamlit Dashboard             <- see dashboard/app.py
```

## Simulation Parameters
| Parameter | Value |
|-----------|-------|
| Rounds | 30 |
| Planets per round | 10 |
| Telescope hours/night | 8 hrs |
| Weather model | AR(1), rho=0.65 |
| Exploration decay tau | 15 rounds |


In [ ]:
# ==============================================================
# CELL 1 - PULL FROM GITHUB  (run at START of every session)
# ==============================================================
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

GITHUB_USER = 'rushikesh-D69'
REPO_NAME   = 'water'
BRANCH      = 'main'
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

if IN_COLAB:
    REPO_PATH = f'/content/{REPO_NAME}'
    if not os.path.exists(REPO_PATH):
        print(f'[Git] Cloning {REPO_URL} ...')
        os.system(f'git clone {REPO_URL} {REPO_PATH}')
    else:
        print('[Git] Repo exists. Pulling latest ...')
        os.system(f'git -C {REPO_PATH} pull origin {BRANCH}')
    os.chdir(REPO_PATH)
    sys.path.insert(0, REPO_PATH)
    print('[Colab] Installing dependencies ...')
    os.system('pip install -q xgboost lightgbm shap scipy scikit-learn matplotlib seaborn requests joblib')
    print('[Colab] Ready.')
else:
    ROOT = os.path.abspath('.')
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    print(f'[Local] Project root: {ROOT}')


---
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Stage 1 pipeline
from src.data_acquisition import run_pipeline, ML_FEATURES, TARGET

# Stage 2 modules
from src.constraint_engine  import ObservationConstraintEngine, WeatherModel
from src.observation_simulator import ObservationSimulator
from src.scheduler import (
    StaticPriorityScheduler,
    DetectabilityGreedyScheduler,
    UncertaintyGreedyScheduler,
    AdaptiveScheduler,
    OracleScheduler,
    run_campaign,
)
from src.evaluation import run_full_evaluation

# Stage 1 ML pipeline for predictions + uncertainty
from src.ml_pipeline import run_ml_pipeline

print('[Setup] All imports OK.')


---
## 2. Load Stage 1 Data & Predictions

In [ ]:
import joblib
from pathlib import Path

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')

# Load processed dataset
df_ml = pd.read_csv(DATA_DIR / 'exoplanets_processed.csv')
print(f'[Data] Loaded {len(df_ml):,} planets | {len(df_ml.columns)} columns')

# Load Stage 1 models for predictions + uncertainty
available_models = list(MODELS_DIR.glob('*.joblib')) + list(MODELS_DIR.glob('*.json'))
print(f'[Models] Found: {[m.name for m in available_models]}')

# Load LightGBM (best Stage 1 model) for predictions
try:
    import lightgbm as lgb
    lgb_model = lgb.Booster(model_file=str(MODELS_DIR / 'lightgbm.joblib'))
    print('[Models] LightGBM loaded')
except Exception as e:
    print(f'[Models] LightGBM load failed: {e} - will rerun ML pipeline')
    lgb_model = None

# Load Random Forest for uncertainty (tree variance)
try:
    rf_model = joblib.load(MODELS_DIR / 'random_forest.joblib')
    print('[Models] Random Forest loaded (for uncertainty)')
except Exception as e:
    print(f'[Models] RF load failed: {e}')
    rf_model = None


In [ ]:
# Get ML feature columns (leakage-free)
FEATURE_COLS = [c for c in ML_FEATURES if c in df_ml.columns]
X = df_ml[FEATURE_COLS].fillna(df_ml[FEATURE_COLS].median()).values

# Predictions from LightGBM (or Random Forest as fallback)
if lgb_model is not None:
    mu_pred = lgb_model.predict(X)
elif rf_model is not None:
    mu_pred = rf_model.predict(X)
else:
    # Fallback: use priority_score directly
    mu_pred = df_ml[TARGET].values
    print('[Warn] Using raw priority_score as mu_pred')

mu_pred = np.clip(mu_pred, 0, 1)

# Uncertainty from Random Forest tree variance
if rf_model is not None:
    tree_preds = np.stack([tree.predict(X) for tree in rf_model.estimators_], axis=0)
    sigma_pred = np.std(tree_preds, axis=0)
else:
    # Fallback: uniform uncertainty
    sigma_pred = np.full(len(df_ml), 0.1)
    print('[Warn] Using uniform sigma=0.1')

sigma_pred = np.clip(sigma_pred, 0.01, 1.0)

# True priority scores (for Oracle scheduler)
true_priorities = df_ml[TARGET].values

print(f'[Predictions] mu:    mean={mu_pred.mean():.4f}  std={mu_pred.std():.4f}')
print(f'[Predictions] sigma: mean={sigma_pred.mean():.4f}  max={sigma_pred.max():.4f}')
print(f'[Oracle]      true:  mean={true_priorities.mean():.4f}  std={true_priorities.std():.4f}')


---
## 3. Initialize Constraint Engine & Observation Simulator

In [ ]:
# Simulation parameters
N_ROUNDS    = 30
K_PER_ROUND = 10
SEED        = 42

print(f'[Config] {N_ROUNDS} rounds x {K_PER_ROUND} planets = {N_ROUNDS * K_PER_ROUND} total observations')
print(f'         from {len(df_ml):,} candidate planets ({N_ROUNDS * K_PER_ROUND / len(df_ml) * 100:.1f}% of pool)')
print(f'         Telescope budget: 8 hrs/night | Weather AR(1) rho=0.65')


---
## 4. Run Observation Campaigns (All 5 Schedulers)

In [ ]:
import copy

def make_fresh_simulator(seed_offset=0):
    return ObservationSimulator(
        df             = df_ml,
        initial_means  = mu_pred.copy(),
        initial_sigmas = sigma_pred.copy(),
        seed           = SEED + seed_offset,
    )

def make_fresh_ce(seed_offset=0):
    return ObservationConstraintEngine(df_ml, seed=SEED + seed_offset)

results = {}

# 1. Static Priority
print('\n[1/5] Static Priority Scheduler ...')
sim1 = make_fresh_simulator(1)
ce1  = make_fresh_ce(1)
s1   = StaticPriorityScheduler(df_ml, true_priorities)  # uses static scores
results['Static Priority'] = run_campaign(s1, sim1, ce1, N_ROUNDS, K_PER_ROUND, verbose=True)

# 2. Detectability Greedy
print('\n[2/5] Detectability Greedy Scheduler ...')
sim2 = make_fresh_simulator(2)
ce2  = make_fresh_ce(2)
s2   = DetectabilityGreedyScheduler('Detectability Greedy', df_ml)
results['Detectability Greedy'] = run_campaign(s2, sim2, ce2, N_ROUNDS, K_PER_ROUND, verbose=True)

# 3. Uncertainty Greedy
print('\n[3/5] Uncertainty Greedy Scheduler ...')
sim3 = make_fresh_simulator(3)
ce3  = make_fresh_ce(3)
s3   = UncertaintyGreedyScheduler('Uncertainty Greedy', df_ml)
results['Uncertainty Greedy'] = run_campaign(s3, sim3, ce3, N_ROUNDS, K_PER_ROUND, verbose=True)

# 4. Adaptive Scheduler (Our Method)
print('\n[4/5] Adaptive Scheduler (Our Method) ...')
sim4 = make_fresh_simulator(4)
ce4  = make_fresh_ce(4)
s4   = AdaptiveScheduler(df_ml, alpha_0=0.50, beta_0=0.30, gamma=0.20, tau=15.0)
results['Adaptive Scheduler'] = run_campaign(s4, sim4, ce4, N_ROUNDS, K_PER_ROUND, verbose=True)

# 5. Oracle Scheduler (Upper Bound)
print('\n[5/5] Oracle Scheduler (Upper Bound) ...')
sim5 = make_fresh_simulator(5)
ce5  = make_fresh_ce(5)
s5   = OracleScheduler(df_ml, true_priorities)
results['Oracle'] = run_campaign(s5, sim5, ce5, N_ROUNDS, K_PER_ROUND, verbose=True)

print('\n[Campaigns] All 5 schedulers complete.')
oracle_cum_gain = results['Oracle']['cumulative_gain']
print(f'[Oracle] Upper bound cumulative gain: {oracle_cum_gain:.4f}')


---
## 5. Evaluation: 7 Metrics + 7 Plots

In [ ]:
# Get weather history from one of the campaigns
weather_history = results.get('Adaptive Scheduler', {}).get('weather_history', [])

comparison_df = run_full_evaluation(
    results         = results,
    n_rounds        = N_ROUNDS,
    k_per_round     = K_PER_ROUND,
    n_planets       = len(df_ml),
    weather_history = weather_history,
    df              = df_ml,
    oracle_cum_gain = oracle_cum_gain,
)

print('\nFull Comparison Table:')
display(comparison_df)


---
## 6. Adaptive Scheduler Deep-Dive

In [ ]:
adaptive_logs = results['Adaptive Scheduler']['logs_df']
oracle_logs   = results['Oracle']['logs_df']
adaptive_obs  = results['Adaptive Scheduler']['obs_history_df']

print('=== Adaptive Scheduler - Round-by-Round Summary ===')
display(adaptive_logs[['round','weather','time_used_hrs','alpha_t','beta_t',
                        'cum_sci_gain','mean_priority','mean_sigma_before',
                        'mean_sigma_after','top_target']].head(30))


In [ ]:
print('=== Top 20 Observations by Scientific Gain (Adaptive) ===')
top_obs = (adaptive_obs
    .assign(sci_gain=lambda d: d['sigma_before'] * d['detectability'])
    .sort_values('sci_gain', ascending=False)
    .head(20)[['round','planet_name','mu_before','sigma_before','mu_after',
               'sigma_after','weather','snr_effective','sci_gain']]
)
display(top_obs)


---
## 7. Regret Analysis vs Oracle

In [ ]:
from src.evaluation import compute_regret

print('=== Final Regret vs Oracle ===')
for name, res in results.items():
    if name == 'Oracle':
        continue
    logs = res['logs_df']
    if logs.empty:
        continue
    regret_series = compute_regret(logs, oracle_cum_gain)
    final_regret  = regret_series.iloc[-1]
    avg_regret    = regret_series.mean()
    print(f'  {name:<25} | Final regret: {final_regret:.4f} | Avg regret: {avg_regret:.4f}')


---
## 8. Campaign Diversity Analysis

In [ ]:
from src.evaluation import compute_campaign_diversity

print('=== Campaign Diversity Scores ===')
for name, res in results.items():
    obs = res['obs_history_df']
    if obs.empty or 'planet_idx' not in obs.columns:
        continue
    obs_idx = obs['planet_idx'].unique().tolist()
    div = compute_campaign_diversity(obs_idx, df_ml)
    print(f'\n  {name}:')
    for k, v in div.items():
        print(f'    {k:<30}: {v:.4f}')


---
## 9. Save Results

In [ ]:
from pathlib import Path
DATA_DIR = Path('data')

# Save comparison table
comparison_df.to_csv(DATA_DIR / 'stage2_comparison.csv', index=False)
print(f'[Save] Comparison table -> data/stage2_comparison.csv')

# Save per-scheduler logs
for name, res in results.items():
    safe_name = name.lower().replace(' ', '_')
    if not res['logs_df'].empty:
        res['logs_df'].to_csv(DATA_DIR / f's2_{safe_name}_logs.csv', index=False)
        print(f'[Save] {name} logs -> data/s2_{safe_name}_logs.csv')

# Save adaptive observation history
adap_obs = results['Adaptive Scheduler']['obs_history_df']
if not adap_obs.empty:
    adap_obs.to_csv(DATA_DIR / 's2_adaptive_obs_history.csv', index=False)
    print('[Save] Adaptive obs history -> data/s2_adaptive_obs_history.csv')

print('\n[Done] Stage 2 complete.')


---
## Push to GitHub

In [ ]:
# ==============================================================
# LAST CELL - PUSH TO GITHUB  (run at END of every session)
# ==============================================================
import os

IN_COLAB = 'google.colab' in sys.modules

GITHUB_USER = 'rushikesh-D69'
REPO_NAME   = 'water'
BRANCH      = 'main'
GIT_EMAIL   = 'rikki0501hanuman@gmail.com'
GIT_NAME    = 'rushikesh-D69'

if IN_COLAB:
    os.system(f'git config user.email "{GIT_EMAIL}"')
    os.system(f'git config user.name "{GIT_NAME}"')
    print('[Git] Pulling latest before push ...')
    os.system(f'git pull origin {BRANCH}')
    print('[Git] Staging all files ...')
    os.system('git add -A')
    status = os.popen('git status --porcelain').read().strip()
    if status:
        print('[Git] Changes detected:')
        print(status)
        os.system('git commit -m "Colab sync: Stage 2 results, plots, data"')
        from google.colab import userdata
        token  = userdata.get('GITHUB_TOKEN')
        remote = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
        os.system(f'git remote set-url origin {remote}')
        print('[Git] Pushing to GitHub ...')
        os.system(f'git push origin {BRANCH}')
        print('[Git] Push complete.')
    else:
        print('[Git] No changes to push.')
else:
    print('[Local] Run: git add -A && git commit -m "msg" && git push')
